In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-core/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Retries, crashes and the reaper

This notebook shows what occurs in three cases:

- The model returns HTTP 503.
- The worker crashes at the worst time.
- Two workers operate on the same run at the same time.

In [ ]:
import sys; sys.path[:0] = [".."]                # the lab root, from notebooks/ or solutions/
from core import Engine, Queue, Store, Crash, drain
from workflow import STEPS, CALLS
clock = [0.0]; now = lambda: clock[0]              # a clock we control: days pass in one line
def fresh(steps=STEPS):
    CALLS.clear(); return Engine(Store(), Queue(), dict(steps), clock=now, lease_ttl=60)


## 1. A flaky step

Model APIs return HTTP 503. For the engine, each `Exception` is retryable. The engine increases the attempt, and then it enqueues a retry with a delay. The increase makes the old task stale by design.

In [ ]:
# TODO: fill in: attempts
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: attempts")
calls = []
def flaky(ctx):
    calls.append(1)
    if len(calls) < 3:
        raise ConnectionError("503 model overloaded")
    return ("done", "ok")

engine = fresh({"flaky": flaky})
engine.start("r", "flaky", {})
print(drain(engine, engine.queue, now), engine.store.get("r")["attempts"], "next due at", engine.queue.tasks[0][0])
clock[0] += 2;  print(drain(engine, engine.queue, now), engine.store.get("r")["attempts"])
clock[0] += 4;  print(drain(engine, engine.queue, now))
assert engine.store.get("r")["attempts"] == {"flaky": ____} and engine.store.get("r")["status"] == "SUCCEEDED"

## 2. Giving up

After `max_attempts`, the run fails with an error that a person can read. What does the run look like?

In [ ]:
# TODO: fill in: status
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: status")
engine = fresh({"bad": lambda ctx: 1 / 0})
engine.start("r", "bad", {})
for _ in range(3):
    drain(engine, engine.queue, now); clock[0] += 10
run = engine.store.get("r")
print(run["status"], "|", run["error"]); print(run["history"])
assert run["status"] == ____ and len(run["history"]) == 3

## 3. The crash the queue cannot see

The worker crashes **after** the checkpoint and **before** it enqueues the next step. Cloud Tasks delivers the old task again, and that task is stale. Nothing enqueues the new task. Which part of the system repairs the run, and when?

In [ ]:
# TODO: fill in: seconds
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: seconds")
engine = fresh()
engine.crash_before_enqueue = True
engine.start("run-1", "draft", {"topic": "x"})
print(drain(engine, engine.queue, now))
run = engine.store.get("run-1"); print("step:", run["step"], "| lease until:", run["lease"])
clock[0] += 30; print("redelivery:", drain(engine, engine.queue, now))
print("reap now:", engine.reap())
clock[0] += ____
print("reap later:", engine.reap())
print(drain(engine, engine.queue, now))
assert engine.store.get("run-1")["status"] == "WAITING"

## 4. Two workers, one run

A second replica receives a duplicate delivery while the first replica is in the middle of a step. The lease makes the second replica refuse the task and return HTTP 503. Then Cloud Tasks retries the task later.

In [ ]:
# TODO: fill in: after
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: after")
engine = fresh()
engine.start("run-1", "draft", {"topic": "x"})
run = engine.store.get("run-1"); run["lease"] = now() + 60; engine.store.save(run)   # replica A holds the lease
print(engine.execute("run-1", "draft", 1))
clock[0] += 61                                                                       # A died; lease expired
assert engine.execute("run-1", "draft", 1) == ____

## 5. Your turn: a step that must never be retried

A failure of a business rule (a customer on a sanctions list) is not an outage. Make `check` end the run as `FAILED`-by-design, **without** the use of retries. (Hint: the failure is an outcome, not an exception.)

In [ ]:
# TODO: fill in: kind
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: kind")
def check(ctx):
    if ctx.input["customer"] in {"acme-blocked"}:
        return (____, {"ok": False, "reason": "sanctions"})
    return ("next", "ship")
engine = fresh({"check": check, "ship": lambda ctx: ("done", {"ok": True})})
engine.start("o1", "check", {"customer": "acme-blocked"})
print(drain(engine, engine.queue, now))
run = engine.store.get("o1")
assert run["status"] == "SUCCEEDED" and run["result"]["ok"] is False and run["attempts"] == {"check": 1}

## 6. A real crash, across two processes

Each crash above was a simulation in one Python process. `workflow.py kill` causes a real crash. It publishes, and then it calls `os._exit(137)` before the checkpoint. No `finally` block runs, and the process does not release the lease.

`workflow.py resume` is a *different* process. An hour later, it opens the same store (`FileStore`, one JSON file that takes the place of Firestore). After that:

- The reaper sees the expired lease.
- The reaper re-drives the step.
- `effect_once` finds the publish record on disk.

In [ ]:
# TODO: fill in: exit_code
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in: exit_code")
import json, shutil, subprocess, sys, tempfile
from pathlib import Path
work = Path(tempfile.mkdtemp())
for name in ("core.py", "workflow.py"):
    shutil.copy(Path("..") / name, work / name)
kill = subprocess.run([sys.executable, "workflow.py", "kill"], cwd=work, capture_output=True, text=True)
print(kill.stdout, "exit code:", kill.returncode)
resume = subprocess.run([sys.executable, "workflow.py", "resume"], cwd=work, capture_output=True, text=True)
print(resume.stdout)
print("effect records on disk:", list(json.loads((work / "runs.json").read_text())["effects"]))
assert kill.returncode == ____ and "publish calls in THIS process: []" in resume.stdout